# 1 · Veri: temizleme, eksiklik, fazlalık, hedef

Teknik raporun 1–4. bölümleri ([technical.tr.md](../reports/technical.tr.md)).

**Ne koşar, ne kayıttan gelir:**
- **Hafif betikler bu defterde koşar.** `script = run("<betik>")` betiğin [1]–[5] hücrelerini çalıştırır: veriyi ve
  kayıtlı model çıktılarını (OOF tahminleri, SHAP dosyası) okur, hesabı yapar; kaydet hücresi [6] çalışmaz.
  Ardından `check` sonucun `metrics/` kaydıyla aynı olup olmadığını gösterir. `script.res`, yüklenen tablolar
  (`script.listings` …) ve betiğin bütün fonksiyonları sonraki hücrelerde kullanılabilir.
- **Ağır betikler kayıttan okunur.** Model eğiten ya da çapraz doğrulama koşan betikler (`HEAVY`) için
  `recorded("<betik>", docs)` `metrics/` kaydını getirir.
- **Soru, veri, yöntem ve sızıntı notları** kartlardan (`analysis/cards/`), **figürler** derleyicilerin çizdiği
  `reports/figures/`'tan gelir.

Analiz kodu betiklerin kendisidir; ortak kod `analysis/lib` ve `builders/report_lib`'den gelir, burada kopyası yok. Bir
sonucu kalıcı olarak değiştirmek için betik değişir ve zincir koşulur; defter yeni kaydı okur. Defter dosya yazmaz.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "analysis" / "lib").is_dir())
sys.path[:0] = [str(ROOT / "analysis"), str(ROOT / "builders")]

import pandas as pd
from IPython.display import Markdown

from lib.labels import lb
from lib.records import card, check, flat, recorded, run, runs, show
from report_lib import metrics_view as MV

pd.set_option("display.max_rows", 30)
pd.set_option("display.max_colwidth", 120)
SCRIPTS = ["01_dedup_leakage", "01_engine_rule", "01_unspecified_panels", "02_missingness", "03_association", "03_segment_quality", "03_brand_ablation", "04_target"]
docs = MV.load_metrics(ROOT / "metrics")
MV.check_consistency(docs)
runs(docs, SCRIPTS)

,yazıldı,veri sonu,DB,run_id
betik,,,,
01_dedup_leakage,2026-11-22T18:55:09+03:00,2026-06-27,444123b1f1a9,None
01_engine_rule,2026-11-22T18:55:10+03:00,2026-06-27,444123b1f1a9,None
01_unspecified_panels,2026-11-22T18:55:14+03:00,2026-06-27,444123b1f1a9,None
02_missingness,2026-11-22T18:55:15+03:00,2026-06-27,444123b1f1a9,None
03_association,2026-11-22T18:55:17+03:00,2026-06-27,444123b1f1a9,None
03_segment_quality,2026-11-22T18:55:19+03:00,2026-06-27,444123b1f1a9,None
03_brand_ablation,2026-09-27T21:32:06+03:00,2026-06-27,444123b1f1a9,None
04_target,2026-11-22T18:55:20+03:00,2026-06-27,444123b1f1a9,None


## 1. Veri temizleme ve sızıntı tespiti

### 01_dedup_leakage
[betik](../analysis/01_dedup_leakage.py) · [kart](../analysis/cards/01_dedup_leakage.json)

In [2]:
Markdown(card("01_dedup_leakage"))

**Soru:** Tarama satırlarından tekil ilana nasıl gelindi, veri hangi sınırlarla toplandı ve fold'lar arasına ne sızabilir?

**Veri:** load_clean(all_snapshots=True) (45.159 TR plakalı tarama satırı) ve load_clean() (29.988 ilan); plaka için veritabanının ham satırları; toplama filtreleri scraper/collection_config.json'dan (salt okunur).

**Yöntem:** Satırlar ad_id başına son kayda indirilir; aynı ilanın taramalar arasındaki fiyat değişimleri sayılır; plaka dağılımı ve eğitime giren ilanlar sayılır; toplama filtreleri koddan okunup verideki karşılıkları (tavan, en eski yıl, yakıt) ölçülür; ad_id'nin göremediği içerik tekrarları katı, gevşek ve fiyatsız tanımla sayılır.

**Bölme:** `none`

**Sızıntı:**
- aynı ilan iki tarafta: `present` — Bu betik ölçümün kendisi: farklı ad_id ile yeniden ilan edilen aynı araç katı tanımda %0,46, fiyatsız tanımda %4,26; modele etkisi ölçülmedi (07_model_comparison kartı).
- hedef öznitelikte: `n/a`
- bütün veriye uydurma: `n/a`
- zamansal: `n/a`

In [3]:
script = run("01_dedup_leakage")           # [1]–[5]: yükle + hesapla; kaydetmez
m = script.to_metrics(script.res)
check(m, docs["01_dedup_leakage"])

{'snapshot_rows': 45159, 'listings': 29988, 'repeat_rows': 15171}
{'listings': 29988, 'seen_again': 10831, 'changed': 4703, 'cuts': 2801, 'rises': 1854, 'returned': 48}


**Kayıtla aynı:** defterde hesaplanan sonuç `metrics/` kaydıyla birebir aynı.

In [4]:
cd, ed = m["methodology"]["content_duplicates"], m["error_drivers"]
show("ölçek", flat(m["meta"]))
show("içerik tekrarları", flat(cd))
show("en çok tekrarlanan içerik", pd.DataFrame(cd["most_repeated"]))
show("fiyat değişimleri (aynı ilan taramalar arasında)", flat(ed["price_changes"]))
show("plaka kapsamı", flat(ed["plate_scope"]))
show("plakaya göre dağılım", pd.DataFrame(ed["plate_scope"]["distribution"]))
show("plakası yüzünden düşen ilanlar", pd.DataFrame(ed["plate_scope"]["dropped_by_plate"]))
show("toplama kapsamı ve verideki ölçümler", flat(ed["scope"]))

**ölçek**

,değer
n_raw,45159
n_dedup,29988
snapshots,"2026-01-18, 2026-01-27, 2026-03-21, 2026-06-27"
brands.bmw,17896
brands.audi,12092


**içerik tekrarları**

,değer
strict_extra,137
strict_pct,0.46
loose_extra,209
loose_pct,0.7
no_price_extra,1278
no_price_pct,4.26
no_price_non_round_km_extra,113
round_km_pct,86.5
n_duplicate_groups,127
strict_key_columns,"price, gb_mileage, gb_year, brand, series, model, kb_fuel, is_heavy_damaged, count_painted, count_changed, power_hp_..."


**en çok tekrarlanan içerik**

,model,repeats,price,year
0,320i Sport Line,4,4900000.0,2025
1,318i Standart,3,717000.0,2005
2,A3 Sportback 1.6 Ambition,3,890000.0,2010
3,116d Joy Plus,3,1044950.0,2015
4,320i First Edition Sport Line,3,2530000.0,2019


... (6 outputs left out)

**plakası yüzünden düşen ilanlar**

,plate,listings
0,(bilinmiyor),54
1,Mavi plakalı,22


**toplama kapsamı ve verideki ölçümler**

,değer
brands,"audi, bmw"
price_min,300000
price_max,6500000
max_km,700000
min_year,2005
fuel_filter,"Benzin, Dizel, Hibrit, LPG"
category,otomobil
measured.price_max,6500000.0
measured.at_cap,10
measured.within_1pct_of_cap,38


### 01_engine_rule
[betik](../analysis/01_engine_rule.py) · [kart](../analysis/cards/01_engine_rule.json)

In [5]:
Markdown(card("01_engine_rule"))

**Soru:** Site hacmi ve gücü aralık olarak verdiğinde model hangi tek sayıyı kullanmalı: alt sınır, orta nokta mı, üst sınır mı?

**Veri:** load_clean() (29.988 ilan): engine_cc_* ve power_hp_* sınırları, aralık bayrağı ve model adı. Fiyat kullanılmıyor.

**Yöntem:** Aralıklı her ilanın üç adayı (alt, orta, üst), aynı modelin kesin değerli ilanlarının medyanıyla karşılaştırılır; kullanıcının seçtiği aday (hacim = üst, güç = orta) en küçük medyan farkı vermezse ya da lib/common.py kuralı değişmişse betik durur. Ayrıca sitenin verdiği bütün (alt, üst) çiftleri ilan sayısıyla yazılır (Figür 30).

**Bölme:** `none`

**Sızıntı:**
- aynı ilan iki tarafta: `none` — load_clean() ad_id başına son kaydı veriyor; her ilan bir kez sayılır.
- hedef öznitelikte: `n/a`
- bütün veriye uydurma: `none` — Model referans medyanları bütün veriden, ama yalnız motor kolonlarından; fiyat hiç okunmuyor, kural fiyattan bilgi taşıyamaz.
- zamansal: `n/a`

In [6]:
script = run("01_engine_rule")           # [1]–[5]: yükle + hesapla; kaydetmez
m = script.to_metrics(script.res)
check(m, docs["01_engine_rule"])

engine_cc {'lower': 194.0, 'mid': 95.5, 'upper': 5.0}
power_hp {'lower': 17.0, 'mid': 7.0, 'upper': 14.0}


**Kayıtla aynı:** defterde hesaplanan sonuç `metrics/` kaydıyla birebir aynı.

In [7]:
er = m["error_drivers"]["engine_rule"]
show("tanım", flat({"definition": er["definition"]}))
for p in ("power_hp", "engine_cc"):
    show(f"{p}: özet", flat(er[p]))
    show(f"{p}: aralık başına kazanan aday (medyan |aday − referans|)",
         pd.DataFrame([{"alt": lo, "üst": up, "ilan": n, "kazanan": w, **{f"fark ({k})": v for k, v in med.items()}}
                       for lo, up, n, w, med in er[p]["range_winners"]]))
    show(f"{p}: sitenin verdiği alt × üst sınır çiftleri",
         pd.DataFrame(er[p]["pairs"], columns=["alt", "üst", "aralık mı", "ilan"]))

**tanım**

,değer
definition,referans = ayni modelin kesin (kovasiz) degerli ilanlarinin medyani; fark = |aday − referans|; yalniz iki siniri da ...


**power_hp: özet**

,değer
unit,hp
chosen,mid
listings,29988
exact,21765
ranged,7797
open_ended,2
no_value,426
with_reference,7148
reference_models,337
inside_pct,84.2


**power_hp: aralık başına kazanan aday (medyan |aday − referans|)**

,alt,üst,ilan,kazanan,fark (lower),fark (mid),fark (upper)
0,101,125,1266,mid,9.0,3.0,15.0
1,126,150,1361,upper,24.0,12.0,0.0
2,151,175,2128,upper,19.0,7.0,5.0
3,176,200,1695,lower,8.0,11.0,23.0
4,201,225,256,lower,3.0,9.0,21.0
5,226,250,132,lower+mid,7.0,7.0,17.0
6,251,275,171,lower,7.0,11.0,23.0


... (4 outputs left out)

**engine_cc: aralık başına kazanan aday (medyan |aday − referans|)**

,alt,üst,ilan,kazanan,fark (lower),fark (mid),fark (upper)
0,1201,1400,123,upper,189.0,89.5,10.0
1,1401,1600,4015,upper,196.0,96.5,3.0
2,1601,1800,146,upper,197.0,97.5,2.0
3,1801,2000,2572,upper,167.0,67.5,32.0
4,2501,3000,248,upper,466.0,216.5,33.0


**engine_cc: sitenin verdiği alt × üst sınır çiftleri**

,alt,üst,aralık mı,ilan
0,999,999,False,317
1,1197,1197,False,59
2,1390,1390,False,219
3,1395,1395,False,706
4,1496,1496,False,869
...,...,...,...,...
55,3001,3500,True,12
56,3501,4000,True,4
57,4001,4500,True,25
58,4501,5000,True,3


![tr-29-hp-cc-rule](../reports/figures/tr-29-hp-cc-rule.png)

![tr-30-engine-bounds](../reports/figures/tr-30-engine-bounds.png)

### 01_unspecified_panels
[betik](../analysis/01_unspecified_panels.py) · [kart](../analysis/cards/01_unspecified_panels.json)

In [8]:
Markdown(card("01_unspecified_panels"))

**Soru:** Kaç panel ve kaç ilan "Belirtilmemiş" (orijinal sayılan) hasar cevabı taşıyor?

**Veri:** load_clean() (29.988 ilan) + data/raw altındaki ham details.jsonl'lerin Hasar_Listesi alanı (salt okunur).

**Yöntem:** Ham panel cevapları ilanlara (ad_id + tarama) bire bir bağlanır; yeniden kurulan bayraklar veritabanının bayraklarıyla satır satır aynı değilse betik durur. Sonra belirtilmemiş panel ve ilan sayılır. Fiyat karşılaştırması yok (2026-09-25'te kaldırıldı).

**Bölme:** `none`

**Sızıntı:**
- aynı ilan iki tarafta: `n/a`
- hedef öznitelikte: `n/a`
- bütün veriye uydurma: `n/a`
- zamansal: `n/a`

In [9]:
script = run("01_unspecified_panels")           # [1]–[5]: yükle + hesapla; kaydetmez
m = script.to_metrics(script.res)
check(m, docs["01_unspecified_panels"])

{'matched': 29988, 'at_least_one': 6236, 'all': 1235}


**Kayıtla aynı:** defterde hesaplanan sonuç `metrics/` kaydıyla birebir aynı.

In [10]:
show("belirtilmemiş paneller", flat(m["error_drivers"]["unspecified"]))

**belirtilmemiş paneller**

,değer
structure.panel,13
structure.flags,39
structure.n_answers,5
structure.grouped_panels.bumper,2
structure.grouped_panels.door,4
structure.grouped_panels.fender,4
structure.single_panels,3
matched_listings,29988
listings,29988
duplicates_skipped,0


## 2. Eksiklik rastgele değil

### 02_missingness
[betik](../analysis/02_missingness.py) · [kart](../analysis/cards/02_missingness.json)

In [11]:
Markdown(card("02_missingness"))

**Soru:** Hangi kolonlar eksik, hangileri birlikte eksik; her ham kolon modele neden girdi ya da neden atıldı?

**Veri:** load_clean(derived=False) (29.988 ilan, ham veritabanı kolonları) ve db/gold_rules.json (belirtilmemiş kolonları ayırmak için, salt okunur).

**Yöntem:** Kolon başına eksik payı; çok eksik kolonlar birlikte-eksik payıyla bloklara kümelenir (blok payı %90'ın altına düşerse durur); her ham kolon öncelik sırasıyla tek bir sınıfa atanır (modelde > hedef > kimlik > türetilmiş > ikiz > yarı-sabit > %40+ eksik > katalog bloğu > gerekçesiz); kb/gb ikizleri karşılaştırılır; belirtilmemiş (gold kurallı) kolonlar eksik listesinin dışında ayrı sayılır.

**Bölme:** `none`

**Sızıntı:**
- aynı ilan iki tarafta: `n/a`
- hedef öznitelikte: `none` — Fiyat yalnız 'hedef' sınıfına atanıyor; hiçbir öznitelik fiyattan türemiyor.
- bütün veriye uydurma: `none` — Hangi kolonun modele gireceği bütün veriden (eksiklik, sabitlik) seçiliyor, ama fiyat kullanılmadan; hedef bilgisi taşımaz.
- zamansal: `n/a`

In [12]:
script = run("02_missingness")           # [1]–[5]: yükle + hesapla; kaydetmez
m = script.to_metrics(script.res)
check(m, docs["02_missingness"])

{'model': 53, 'target': 1, 'C': 8, 'F': 9, 'B': 11, 'A': 4, 'D': 5, 'E': 22, 'G': 4} | blocks: [16, 2, 2, 2]


**Kayıtla aynı:** defterde hesaplanan sonuç `metrics/` kaydıyla birebir aynı.

In [13]:
me = m["methodology"]
sm = me["systematic_missing"]
show("modele giren öznitelikler: eksik oranı", pd.DataFrame(me["column_missing"], columns=["kolon", "eksik %"]))
show("bütün kolonlar: eksik oranı", pd.DataFrame(me["column_missing_all"], columns=["kolon", "eksik %"]))
show("birlikte eksik gruplar", pd.DataFrame(sm["systematic_groups"]))
show("belirtilmemiş kolonlar: özet", flat(sm["unspecified"]))
show("belirtilmemiş kolonlar", pd.DataFrame(sm["unspecified"]["columns"], columns=["kolon", "belirtilmemiş %"]))
show("modele girmeyen kolonlar", pd.DataFrame([[c, why, n, cols] for c, why, n, cols, _en in me["feature_drop"]],
                                             columns=["sınıf", "gerekçe", "kolon sayısı", "kolonlar"]))
show("kolon hesabı", flat(me["column_accounting"]))
show("kb / gb ikizleri", pd.DataFrame(me["kb_gb_twins"]))
show("notlar", flat({"impute_note": me["impute_note"], "systematic_note": sm["note"]}))

**modele giren öznitelikler: eksik oranı**

,kolon,eksik %
0,kb_drivetrain,1.5
1,power_hp_val,1.4
2,engine_cc_val,1.2
3,kb_body_type,1.0
4,model,0.0
5,series,0.0
6,brand,0.0
7,segment,0.0
8,kb_transmission,0.0
9,kb_fuel,0.0


**bütün kolonlar: eksik oranı**

,kolon,eksik %
0,tramer_fee,84.4
1,transmission_brand,74.1
2,gb_drivetrain,74.0
3,gb_traffic_insurance_avg,54.3
4,gb_kasko_avg,51.1
...,...,...
27,seat_count,27.5
28,gb_warranty_status,22.1
29,kb_trade_available,19.1
30,gb_trade_available,19.1


**birlikte eksik gruplar**

,n_columns,mean_missing_pct,co_missing_pct,sample_columns,columns
0,16,27.6,98.8,"[weight_kg, kb_fuel_cons_avg, kb_fuel_tank, gb_segment, torque_nm, cylinder_count, max_speed_kmh, accel_0_100]","[weight_kg, kb_fuel_cons_avg, kb_fuel_tank, gb_segment, torque_nm, cylinder_count, max_speed_kmh, accel_0_100, lengt..."
1,2,31.4,100.0,"[city_fuel_cons, highway_fuel_cons]","[city_fuel_cons, highway_fuel_cons]"
2,2,29.9,100.0,"[production_year_start, production_year_end]","[production_year_start, production_year_end]"
3,2,29.3,100.0,"[rpm_max, rpm_min]","[rpm_max, rpm_min]"


... (8 outputs left out)

**kb / gb ikizleri**

,kb,gb,kb_missing_pct,gb_missing_pct,kb_unique,gb_unique,same_pct,kept
0,kb_year,gb_year,0.0,0.0,22,22,100.0,gb_year
1,kb_mileage,gb_mileage,0.0,0.0,3109,3109,100.0,gb_mileage
2,kb_transmission,gb_transmission,0.0,0.0,3,3,100.0,kb_transmission
3,kb_fuel,gb_fuel,0.0,0.0,4,4,100.0,kb_fuel
4,kb_body_type,gb_body_type,1.0,0.7,9,23,77.5,kb_body_type
5,kb_color,gb_color,0.0,0.0,24,24,100.0,NaN
6,kb_drivetrain,gb_drivetrain,1.5,74.0,4,4,27.5,kb_drivetrain
7,kb_condition,gb_condition,0.0,0.0,2,2,100.0,NaN
8,kb_trade_available,gb_trade_available,19.1,19.1,2,2,100.0,NaN
9,kb_seller_type,gb_seller_type,0.0,0.0,3,3,100.0,NaN


**notlar**

,değer
impute_note,Sayısal özniteliklerde doldurma yapılmadı: eksik değerler LightGBM/CatBoost'a NaN olarak girer ve kütüphanenin kendi...
systematic_note,"Katalog eşleştirmesi: standart modeller eşleşir, özel varyantlar eşleşmez → tüm spec birden boş. Gruplar birlikte-ek..."


![tr-16-missing](../reports/figures/tr-16-missing.png)

## 3. Fazlalık, bağıntı ve marka

### 03_association
[betik](../analysis/03_association.py) · [kart](../analysis/cards/03_association.json)

In [14]:
Markdown(card("03_association"))

**Soru:** Kategorik öznitelikler birbirine hangi yönde bağlı; sayısal öznitelikler arasında hangi çiftler birlikte hareket ediyor?

**Veri:** load_clean() (29.988 ilan): 11 kategorik ve 14 sayısal öznitelik.

**Yöntem:** Kategorikler için Theil's U (yönlü); sayısallar için Spearman, |ρ|>0,5 çiftleri işaretlenir.

**Bölme:** `none`

**Sızıntı:**
- aynı ilan iki tarafta: `n/a`
- hedef öznitelikte: `n/a`
- bütün veriye uydurma: `n/a`
- zamansal: `n/a`

In [15]:
script = run("03_association")           # [1]–[5]: yükle + hesapla; kaydetmez
m = script.to_metrics(script.res)
check(m, docs["03_association"])

high numeric pairs | yüksek sayısal çiftler: [['vehicle_age', 'gb_mileage', 0.736], ['power_hp_val', 'engine_cc_val', 0.606], ['door_painted', 'fender_painted', 0.603]]


**Kayıtla aynı:** defterde hesaplanan sonuç `metrics/` kaydıyla birebir aynı.

In [16]:
th, nc = m["methodology"]["theils_matrix"], m["domain"]["numeric_correlation"]
show("Theil's U (satır, sütunu ne kadar açıklıyor)",
     pd.DataFrame(th["matrix"], index=[lb(c, "tr") for c in th["labels"]], columns=[lb(c, "tr") for c in th["labels"]]))
show("Spearman", pd.DataFrame(nc["spearman"], index=[lb(c, "tr") for c in nc["labels"]],
                              columns=[lb(c, "tr") for c in nc["labels"]]))
show("yüksek bağıntılı çiftler", pd.DataFrame(nc["high_pairs"], columns=["kolon 1", "kolon 2", "Spearman"]))

**Theil's U (satır, sütunu ne kadar açıklıyor)**

,Marka,Kasa tipi,Çekiş,Segment,Vites,Yakıt,Tavan,Kaput,Bagaj,model,series
Marka,1.000,0.061,0.683,0.071,0.346,0.088,0.000,0.001,0.000,1.000,1.000
Kasa tipi,0.040,1.000,0.063,0.270,0.040,0.017,0.001,0.004,0.002,0.892,0.604
Çekiş,0.442,0.062,1.000,0.160,0.177,0.095,0.000,0.001,0.001,0.932,0.627
Segment,0.041,0.235,0.142,1.000,0.050,0.030,0.001,0.002,0.001,0.999,0.997
Vites,0.337,0.059,0.267,0.085,1.000,0.071,0.002,0.007,0.007,0.720,0.387
Yakıt,0.063,0.019,0.105,0.037,0.052,1.000,0.001,0.015,0.007,0.879,0.119
Tavan,0.001,0.010,0.007,0.009,0.017,0.020,1.000,0.208,0.232,0.232,0.022
Kaput,0.001,0.005,0.001,0.003,0.006,0.018,0.018,1.000,0.046,0.104,0.009
Bagaj,0.001,0.004,0.003,0.002,0.011,0.014,0.036,0.081,1.000,0.122,0.006
model,0.125,0.169,0.180,0.217,0.092,0.154,0.003,0.016,0.010,1.000,0.387


**Spearman**

,Yaş (yıl),Kilometre,Motor gücü (hp),Motor hacmi (cc),Kapı değişen,Kapı boyalı,Kapı lokal,Çamurluk değişen,Çamurluk boyalı,Çamurluk lokal,Tampon değişen,Tampon boyalı,Tampon lokal,Ağır hasar
Yaş (yıl),1.000,0.736,-0.106,0.334,0.119,0.269,0.088,0.201,0.310,0.118,0.029,0.157,0.051,0.057
Kilometre,0.736,1.000,0.077,0.409,0.104,0.272,0.102,0.217,0.303,0.145,0.014,0.120,0.038,0.073
Motor gücü (hp),-0.106,0.077,1.000,0.606,-0.035,-0.050,-0.016,-0.018,-0.056,-0.015,-0.027,-0.042,-0.027,0.017
Motor hacmi (cc),0.334,0.409,0.606,1.000,0.039,0.117,0.039,0.096,0.126,0.052,-0.023,0.031,-0.004,0.043
Kapı değişen,0.119,0.104,-0.035,0.039,1.000,0.129,-0.002,0.218,0.197,0.047,-0.003,0.039,-0.002,0.050
Kapı boyalı,0.269,0.272,-0.050,0.117,0.129,1.000,-0.047,0.267,0.603,0.056,0.002,0.177,-0.015,0.113
Kapı lokal,0.088,0.102,-0.016,0.039,-0.002,-0.047,1.000,0.049,0.005,0.380,0.000,-0.021,0.059,0.006
Çamurluk değişen,0.201,0.217,-0.018,0.096,0.218,0.267,0.049,1.000,0.167,0.032,0.047,0.055,0.011,0.147
Çamurluk boyalı,0.310,0.303,-0.056,0.126,0.197,0.603,0.005,0.167,1.000,-0.028,0.002,0.192,-0.011,0.126
Çamurluk lokal,0.118,0.145,-0.015,0.052,0.047,0.056,0.380,0.032,-0.028,1.000,-0.016,-0.040,0.058,-0.000


**yüksek bağıntılı çiftler**

,kolon 1,kolon 2,Spearman
0,vehicle_age,gb_mileage,0.736
1,power_hp_val,engine_cc_val,0.606
2,door_painted,fender_painted,0.603


![tr-21-spearman](../reports/figures/tr-21-spearman.png)

### 03_segment_quality
[betik](../analysis/03_segment_quality.py) · [kart](../analysis/cards/03_segment_quality.json)

In [17]:
Markdown(card("03_segment_quality"))

**Soru:** Segment beslemeden mi alınmalı, türetilmeli mi; türetilen segment nasıl çözülüyor ve beslemeyle ne kadar ayrışıyor?

**Veri:** load_clean(derived=False) ve load_clean() (29.988 ilan); segment kuralı analysis/lib/segment_rule.py.

**Yöntem:** Her ilanın segmenti seri haritasından, model adı tablosundan ya da temel sınıftan çözülür (çözülemeyen kalırsa durur); G segmentinin içeriği, model adından çözülen aileler, birden çok segmente yayılan seriler ve ham segmentle uyuşmazlık sayılır; seri × segment medyan fiyat matrisi.

**Bölme:** `none`

**Sızıntı:**
- aynı ilan iki tarafta: `n/a`
- hedef öznitelikte: `n/a`
- bütün veriye uydurma: `n/a`
- zamansal: `n/a`

In [18]:
script = run("03_segment_quality")           # [1]–[5]: yükle + hesapla; kaydetmez
m = script.to_metrics(script.res)
check(m, docs["03_segment_quality"])

raw ≠ derived | ham ≠ türetilen: 404 / 21723


**Kayıtla aynı:** defterde hesaplanan sonuç `metrics/` kaydıyla birebir aynı.

In [19]:
sq = m["error_drivers"]["segment_quality"]
show("seri × segment", pd.DataFrame(m["domain"]["series_segment_matrix"], columns=["seri", "segment", "medyan fiyat", "ilan"]))
show("segment kuralı: özet", flat(sq))
show("G segmenti: kasa tipi", pd.DataFrame(sq["g_segment"]["body"], columns=["kasa tipi", "ilan"]))
show("G segmenti: seri", pd.DataFrame(sq["g_segment"]["series"], columns=["seri", "ilan"]))
show("segmenti model adından çıkan seriler",
     pd.DataFrame([[s_, n, ", ".join(f"{k}: {v}" for k, v in segs)] for s_, n, segs in sq["from_model_name"]],
                  columns=["seri", "ilan", "segmentler"]))
show("ham segment ≠ türetilen", pd.DataFrame(sq["mismatch"]["cross"], columns=["ham", "türetilen", "ilan"]))

**seri × segment**

,seri,segment,medyan fiyat,ilan
0,1 Serisi,C,1150000.0,2725
1,2 Serisi,C,2050000.0,789
2,3 Serisi,D,1425000.0,7307
3,4 Serisi,D,2089500.0,934
4,5 Serisi,E,1855000.0,5687
5,6 Serisi,E,3040000.0,51
6,7 Serisi,F,2740000.0,244
7,A1,B,939500.0,127
8,A3,C,1350000.0,5625
9,A4,D,1450000.0,3046


**segment kuralı: özet**

,değer
map_size,18
g_segment.n,215
g_segment.mpv_n,214
paths.series_map,29832
paths.base_class,137
paths.model_table,19
multi_segment_series,"M Serisi, RS, S"
mismatch.raw_known,21723
mismatch.differs,404
mismatch.pct,1.86


**G segmenti: kasa tipi**

,kasa tipi,ilan
0,MPV,214
1,Station wagon,1


... (2 outputs left out)

**segmenti model adından çıkan seriler**

,seri,ilan,segmentler
0,M Serisi,82,"D: 44, E: 29, C: 8, S: 1"
1,RS,2,"D: 1, E: 1"
2,S,54,"C: 43, D: 8, F: 3"
3,i Serisi,18,S: 18


**ham segment ≠ türetilen**

,ham,türetilen,ilan
0,G,C,215
1,E,D,95
2,E,S,61
3,F,E,32
4,F,S,1


![tr-19-series-segment](../reports/figures/tr-19-series-segment.png)

### 03_brand_ablation
[betik](../analysis/03_brand_ablation.py) · [kart](../analysis/cards/03_brand_ablation.json)

In [20]:
Markdown(card("03_brand_ablation"))

**Soru:** Marka, seri ve modelin yanında ayrı bilgi taşıyor mu?

**Veri:** load_clean() (29.988 ilan), manşet modelin öznitelikleri.

**Yöntem:** Manşet LightGBM üç kolla, aynı 5-fold OOF ve parametrelerle koşulur: yalnız marka (seri, model ve segment çıkar), seri + model, marka + seri + model (manşet model). Tesisat kapısı: marka CV modellerinde bölme kazanıp da OOF birebir aynı kalırsa durur.

**Bölme:** `kfold_oof`

**Sızıntı:**
- aynı ilan iki tarafta: `present` — Farklı ad_id ile yeniden ilan edilen aynı araç fold'lar arasına düşebilir: 01_dedup_leakage katı tanımda %0,46, fiyatsız tanımda %4,26 ölçüyor; etkisi ölçülmedi.
- hedef öznitelikte: `none` — Öznitelik listesinde (lib/common.FEATURES) fiyat ya da fiyattan türeyen bir alan yok.
- bütün veriye uydurma: `present` — Kategori kodları, TF-IDF/SVD ve tabanlar yalnız eğitim fold'unda kurulur (lib/cv.fold_matrices). Ama LightGBM'in erken durdurması her fold'un kendi doğrulama kısmında (lib/cv.fit_lgb_fold): ağaç sayısı skorlanan veriyle seçiliyor, OOF hatası hafifçe iyimser olabilir. Etkisi ölçülmedi (iç içe bölmeyle ölçülebilir).
- zamansal: `present` — KFold dönemleri karıştırır; model zamanı görmez. Medyan artık dönemlere göre kayıyor (08_residuals); ileriye dönük hata 09_backtest'te ayrıca ölçülüyor.

In [21]:
m = recorded("03_brand_ablation", docs)

**Kayıttan okunuyor** (ağır betik: üç kolda 5-fold çapraz doğrulama).

In [22]:
ba = m["domain"]["brand_ablation"]
show("kollar (aynı 5-fold OOF)", pd.DataFrame({arm: ba[arm] for arm in ("brand_only", "series_model", "brand_series_model")}).T)
show("doğrulama ve not", flat({"validation": ba["validation"], "note": ba["note"]}))

**kollar (aynı 5-fold OOF)**

,MAE,MedAE,RMSE,MAPE,R2
brand_only,162812.0,99426.0,267224.0,8.80,0.9413
series_model,109775.0,75320.0,176225.0,6.49,0.9745
brand_series_model,109776.0,75320.0,176225.0,6.49,0.9745


**doğrulama ve not**

,değer
validation.brand_split_cv,26
validation.oof_same,False
validation.max_gap_tl,7856.33
note,"TAM MODELDE kimlik ablasyonu: marka/seri/model kolonları değişir; ""yalnız marka"" kolu seri ve model adından türetile..."


## 4. Hedef ve önişleme

### 04_target
[betik](../analysis/04_target.py) · [kart](../analysis/cards/04_target.json)

In [23]:
Markdown(card("04_target"))

**Soru:** Fiyat nasıl dağılıyor ve model neden log fiyatı hedef alıyor?

**Veri:** load_clean() (29.988 ilan), yalnız fiyat.

**Yöntem:** P10 / medyan / P90, 250 bin TL'lik kovalarla histogram, ham ve log fiyatın çarpıklığı.

**Bölme:** `none`

**Sızıntı:**
- aynı ilan iki tarafta: `n/a`
- hedef öznitelikte: `n/a`
- bütün veriye uydurma: `n/a`
- zamansal: `n/a`

In [24]:
script = run("04_target")           # [1]–[5]: yükle + hesapla; kaydetmez
m = script.to_metrics(script.res)
check(m, docs["04_target"])

{'skew_raw': 1.6249902867591544, 'skew_log': 0.2836429495479539, 'median': 1545000.0, 'p10': 845000.0, 'p90': 3425000.0}


**Kayıtla aynı:** defterde hesaplanan sonuç `metrics/` kaydıyla birebir aynı.

In [25]:
t = m["domain"]
show("fiyat dağılımı", flat(t["price_dist"]))
show("fiyat histogramı", pd.DataFrame(t["price_histogram"], columns=["kova sol kenarı (₺)", "ilan"]))

**fiyat dağılımı**

,değer
skew_raw,1.62499
skew_log,0.283643
median,1545000.0
p10,845000.0
p90,3425000.0


**fiyat histogramı**

,kova sol kenarı (₺),ilan
0,0.0,0
1,250000.0,180
2,500000.0,1643
3,750000.0,3499
4,1000000.0,3883
5,1250000.0,5158
6,1500000.0,3764
7,1750000.0,2602
8,2000000.0,1571
9,2250000.0,1442


![tr-25-price-hist](../reports/figures/tr-25-price-hist.png)